####DAY 14 – Final Production-Ready System
####🏗️ Architecture & Strategy
Welcome to the grand finale of your Databricks AI Challenge! Over the last 13 days, you have built the individual components of a Data Lakehouse: Delta optimizations, Feature Engineering, Hyperparameter Tuning, and MLflow Model Registries.

Today, we transition from "Notebook Engineering" to "Software Engineering."

In a real production environment, you do not run cells manually. You orchestrate your code using Databricks Workflows. To do this, your code must be combined, fault-tolerant, and self-contained.

####Our Senior-Level Strategy:

The Unified Pipeline: We will combine the Data Pipeline (Feature Engineering) and the ML Pipeline (Model Training) into a single, cohesive PySpark Pipeline.

* **Failure Handling (try-except)**: Production systems fail. A Senior Engineer anticipates this. We will wrap our pipeline in a robust error-handling block to ensure that if the data is corrupted, the system alerts us instead of silently failing.

* **Automated MLOps**: We will automate the entire MLflow lifecycle inside this function—training, logging, evaluating, and registering the final model to Unity Catalog.

* **Presentation**: We will execute this unified system and present the final outputs, proving the pipeline is ready for scheduled execution.

####Define the Production Pipeline Engine
Let's encapsulate our entire end-to-end logic into a robust Python function. This is the exact pattern you would use when deploying a Python Wheel or a Databricks Workflow Task.

In [0]:
import os
import time
import mlflow
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from mlflow.models.signature import infer_signature
from mlflow.tracking import MlflowClient

# ---------------------------------------------------------
# GLOBAL CONFIGURATION & SECURITY
# ---------------------------------------------------------
CATALOG_NAME = "course_catalog"  
SCHEMA_NAME = "ecommerce_governed"
VOLUME_NAME = "ml_assets"
MLFLOW_TMP_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}/mlflow_staging"
MODEL_REGISTRY_NAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.purchase_prediction_classifier"

# Force Serverless UC Security Compliance
os.environ["MLFLOW_DFS_TMP"] = MLFLOW_TMP_PATH
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"USE SCHEMA {SCHEMA_NAME}")

def run_production_ml_pipeline(train_table: str, test_table: str):
    """
    End-to-End automated ML pipeline with integrated failure handling and MLflow tracking.
    """
    print(f"⚙️ INITIATING PRODUCTION PIPELINE...")
    start_time = time.time()
    
    try:
        # 1. DATA INGESTION (Silver to Gold Transition)
        print(f"   ➤ Loading Datasets: {train_table} & {test_table}")
        train_df = spark.table(train_table)
        test_df = spark.table(test_table)
        
        # 2. FEATURE ENGINEERING + MODEL ARCHITECTURE
        print("   ➤ Constructing PySpark ML Pipeline...")
        feature_cols = ["total_events", "view_count", "cart_count"]
        assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
        
        # We use Random Forest for its stability and explainability in production
        rf = RandomForestClassifier(featuresCol="features", labelCol="purchased", maxDepth=5, numTrees=50, seed=42)
        
        # Combine Data + ML into one strict pipeline
        prod_pipeline = Pipeline(stages=[assembler, rf])
        evaluator = BinaryClassificationEvaluator(labelCol="purchased", metricName="areaUnderROC")
        
        # 3. MLOPS: MLFLOW TRACKING
        # ⚠️ THE FIX: Use Spark SQL to safely get the username on Serverless compute!
        username = spark.sql("SELECT current_user()").first()[0]
        mlflow.set_experiment(f"/Users/{username}/Day14_Production_System")
        
        with mlflow.start_run(run_name="Automated_Production_Run") as run:
            print("   ➤ Training Distributed Model...")
            pipeline_model = prod_pipeline.fit(train_df)
            
            print("   ➤ Evaluating on Holdout Data...")
            predictions = pipeline_model.transform(test_df)
            auc_score = evaluator.evaluate(predictions)
            
            # Monitoring Check: Did the model degrade?
            if auc_score < 0.80:
                raise ValueError(f"Model AUC ({auc_score:.4f}) dropped below production threshold (0.80)! Aborting deployment.")
            
            mlflow.log_metric("production_auc", auc_score)
            
            print("   ➤ Generating Strict Model Signatures...")
            input_example = train_df.select(feature_cols).limit(1).toPandas()
            output_example = predictions.select("prediction").limit(1).toPandas()
            signature = infer_signature(input_example, output_example)
            
            print("   ➤ Logging and Registering to Unity Catalog...")
            mlflow.spark.log_model(
                spark_model=pipeline_model, 
                artifact_path="prod_rf_pipeline", 
                dfs_tmpdir=MLFLOW_TMP_PATH,
                signature=signature,
                input_example=input_example,
                registered_model_name=MODEL_REGISTRY_NAME # Auto-registers the model
            )
            
            # 4. CHAMPION ALIAS ASSIGNMENT
            client = MlflowClient()
            # Fetch all model versions and select the latest one (by creation time)
            mv_list = client.search_model_versions(f"name='{MODEL_REGISTRY_NAME}'")
            latest_version = max(mv_list, key=lambda v: int(v.version)).version if mv_list else "1"
            client.set_registered_model_alias(name=MODEL_REGISTRY_NAME, alias="champion", version=latest_version)
            
        execution_time = time.time() - start_time
        print(f"\n✅ PIPELINE SUCCESS! (Execution Time: {execution_time:.2f}s)")
        print(f"   🏆 Final Production AUC: {auc_score:.4f}")
        print(f"   👑 Model Version {latest_version} tagged as 'champion' in Unity Catalog.")
        
        return predictions

    except Exception as e:
        # 5. FAILURE HANDLING
        print(f"\n❌ CRITICAL PIPELINE FAILURE:")
        print(f"   Error Details: {str(e)}")
        print("   Alerting MLOps PagerDuty... (Simulated)")
        raise e


####Execute & Present the Complete System
Now, we trigger our orchestration function and natively display the results, proving the entire Databricks Data Intelligence Platform is working in harmony.

In [0]:
# ---------------------------------------------------------
# SYSTEM EXECUTION & PRESENTATION
# ---------------------------------------------------------

# Execute the combined, fault-tolerant pipeline
final_predictions_df = run_production_ml_pipeline(
    train_table="gold_train_set", 
    test_table="gold_test_set"
)

# Present the Business Value: Displaying the final AI-augmented data
print("\n📊 PRESENTING COMPLETE SYSTEM OUTPUT (Sample Predictions):")
display(
    final_predictions_df.select(
        "user_id", 
        "total_events", 
        "cart_count", 
        "purchased", 
        "prediction", 
        "probability"
    ).filter("purchased = 1").limit(10)
)

####Key Learnings & Interview Talking Points
As you wrap up this challenge and update your resume, here is how you articulate your Senior-level architecture capabilities to technical recruiters:

* **Unified PySpark Pipelines**: "I don't leave my feature engineering and model training as disconnected scripts. I combine them using the pyspark.ml.Pipeline API. By deploying the Data processing and ML execution as a single, serialized artifact, I eliminate training-serving skew and guarantee that the production REST API applies the exact same transformations as my training notebook."

* **Fault Tolerance & Defensive Programming**: "Production pipelines fail—data schemas change, upstream tables drop, or model metrics degrade. I wrap my orchestrations in robust try-except blocks. Furthermore, I implement explicit validation checks (e.g., aborting the MLflow registration if the AUC drops below a threshold), ensuring that degraded models are never blindly promoted to production."

* **Scalability & Orchestration Thinking**: "While I develop in Databricks interactive notebooks, I write my final code using modular, functional paradigms. This ensures my architecture is 'lift-and-shift' ready to be orchestrated by Databricks Workflows or Apache Airflow as automated, serverless batch jobs."

* **End-to-End Governance**: "Throughout this project, I demonstrated a complete mastery of the Lakehouse. I ingested raw data to Delta Lake, applied strict Medallion architectures, solved massive class imbalances, secured artifacts using Unity Catalog Volumes, and maintained strict CI/CD aliasing for my ML models using the MLflow Model Registry.